In [1]:
!pip install trl bitsandbytes>=0.46.1 -q

In [10]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
 AutoTokenizer,
 AutoModelForCausalLM,
 BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer


In [3]:
MODEL_C = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer_c = AutoTokenizer.from_pretrained(MODEL_C)
if tokenizer_c.pad_token is None:
 tokenizer_c.pad_token = tokenizer_c.eos_token

In [4]:
use_qlora = torch.cuda.is_available()
if use_qlora:
 compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
 quant_config = BitsAndBytesConfig(
 load_in_4bit=True,
 bnb_4bit_quant_type="nf4",
 bnb_4bit_use_double_quant=True,
 bnb_4bit_compute_dtype=compute_dtype,
 )
 base_c = AutoModelForCausalLM.from_pretrained(
 MODEL_C,
 quantization_config=quant_config,
 device_map="auto",)

 base_c = prepare_model_for_kbit_training(base_c)
else:
 base_c = AutoModelForCausalLM.from_pretrained(MODEL_C)
lora_config = LoraConfig(
  task_type=TaskType.CAUSAL_LM,
  r=8,
  lora_alpha=16,
  lora_dropout=0.05,
  target_modules=["q_proj", "v_proj"],
  bias="none",
)
model_c = get_peft_model(base_c, lora_config)
model_c.print_trainable_parameters()


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


In [11]:
sft_dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/support_specialist_sft_train_48.jsonl",
        "validation": "/content/support_specialist_sft_validation_12.jsonl",
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [13]:
def format_for_sft(example):
 return {
 "text": tokenizer_c.apply_chat_template(
 example["messages"],
 tokenize=False,
 add_generation_prompt=False,
 )
 }
sft_train = sft_dataset["train"].map(format_for_sft)
sft_val = sft_dataset["validation"].map(format_for_sft)
sft_args = SFTConfig(
 output_dir="models/support_adapter",
 num_train_epochs=10,
 per_device_train_batch_size=2,
 per_device_eval_batch_size=2,
 learning_rate=2e-4,
 eval_strategy="epoch",
 save_strategy="epoch",
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 dataset_text_field="text",
 max_length=512,
 packing=False,
 report_to="none",
)
trainer_c = SFTTrainer(
 model=model_c,
 args=sft_args,
 train_dataset=sft_train,
 eval_dataset=sft_val,
 processing_class=tokenizer_c,
)
trainer_c.train()
model_c.save_pretrained("models/support_adapter")
tokenizer_c.save_pretrained("models/support_adapter")

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,3.212718,2.946481,2.954622,6779.000000,0.464678
2,2.811266,2.425231,2.571369,13558.000000,0.568033
3,2.286895,2.141696,2.259497,20337.000000,0.620372
4,2.057426,2.070079,2.120767,27116.000000,0.631350
5,2.030439,2.038815,2.087117,33895.000000,0.632576
6,2.031028,2.022758,2.059186,40674.000000,0.631490
7,2.012438,2.013329,2.049105,47453.000000,0.630987
8,2.002886,2.008471,2.038464,54232.000000,0.633878
9,2.042354,2.006945,2.035891,61011.000000,0.633757
10,2.036767,2.006600,2.036042,67790.000000,0.632696


('models/support_adapter/tokenizer_config.json',
 'models/support_adapter/chat_template.jinja',
 'models/support_adapter/tokenizer.json')